PULocationID = Pull Up Location ID <br>
DOLocatinID = Drop Out Location ID


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

In [2]:
df = pd.read_parquet('../data/yellow_tripdata_2024-01.parquet')
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
print(df['tpep_pickup_datetime'].dt.year.unique())

[2024 2023 2009 2002]


There are data other than 2024

In [6]:
df = df[(df['tpep_pickup_datetime'] >= '2024-01-01') & (df['tpep_pickup_datetime'] < '2024-02-01')]

In [7]:
df.isna().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          140162
trip_distance                 0
RatecodeID               140162
store_and_fwd_flag       140162
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     140162
Airport_fee              140162
dtype: int64

In [8]:
# fill missing passenger with 1
df['passenger_count'] = df['passenger_count'].fillna(1)

In [9]:
# remove illogical trips (zero distance or negative fares)
df = df[(df['trip_distance'] > 0) & (df['fare_amount'] >= 0)]

In [10]:
# remove trips where pickup and dropoff happen at the exact same second
df = df[df['tpep_dropoff_datetime'] > df['tpep_pickup_datetime']]

# Calculate Centroids for Map Projection and Zone Names

In [11]:
gdf = gpd.read_file('../data/taxi_zones/taxi_zones.shp')


In [12]:
gdf = gdf.to_crs(epsg=4326) # convert into standard GPS coordinates for Uber H3

In [13]:
# Calculate the exact center (centroid) of each taxi zone polygon
gdf['longitude'] = gdf.geometry.centroid.x
gdf['latitude'] = gdf.geometry.centroid.y

/tmp/ipykernel_55969/3371146002.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['longitude'] = gdf.geometry.centroid.x
/tmp/ipykernel_55969/3371146002.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['latitude'] = gdf.geometry.centroid.y


In [14]:
# View the mapping
display(gdf[['LocationID', 'zone', 'latitude', 'longitude']].head())

,LocationID,zone,latitude,longitude
0,1,Newark Airport,40.691831,-74.174000
1,2,Jamaica Bay,40.616745,-73.831299
2,3,Allerton/Pelham Gardens,40.864474,-73.847422
3,4,Alphabet City,40.723752,-73.976968
4,5,Arden Heights,40.552659,-74.188484


In [15]:
# Create a fast lookup dictionary: {LocationID: {'lat': latitude, 'lon': longitude}}
zone_lookup = gdf.set_index('LocationID')[['latitude', 'longitude']].to_dict('index')

sample_pickup_id = df.iloc[0]['PULocationID']
sample_dropoff_id = df.iloc[0]['DOLocationID']

print(f"Pickup Zone {sample_pickup_id} Coordinates: {zone_lookup.get(sample_pickup_id)}")
print(f"Dropoff Zone {sample_dropoff_id} Coordinates: {zone_lookup.get(sample_dropoff_id)}")

Pickup Zone 186 Coordinates: {'latitude': 40.748497181405014, 'longitude': -73.9924375369761}
Dropoff Zone 79 Coordinates: {'latitude': 40.72762019590954, 'longitude': -73.98593745682462}
